In [15]:
%pip install sqlalchemy fairscape_models

Defaulting to user installation because normal site-packages is not writeable
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 794.2/794.2 kB 10.6 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.3/2.3 MB 18.2 MB/s  0:00:00
  Created wheel for typing: filename=typing-3.7.4.3-py3-none-any.whl size=26396 sha256=d01739c29bde25fac53b2f8f641c0039dc6dc16757369a06dacbbbf1511d5e51
  Stored in directory: /home/vscode/.cache/pip/wheels/f9/11/0b/6cafb59e5f94f79de21581d27210c9a2849da6affccdf1c23f
Successfully built typing
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8/8 [fairscape_models][fairscape_models]

[notice] A new release of pip is available: 26.1.2 -> 26.2.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [21]:
from fairscape_models.rocrate import (
	ROCrateV1_2,
	ROCrateMetadataElem, 
	ROCrateMetadataFileElem,
)
from fairscape_models.fairscape_base import IdentifierValue

import json
import pathlib
import zipfile

In [3]:
import sqlalchemy as sa
from sqlalchemy.orm import declarative_base, Mapped, mapped_column, relationship
import enum
from pydantic import BaseModel, ConfigDict, Field
from typing import List, Optional

Base = declarative_base()

In [ ]:
class MetadataTypeEnumSQL(enum.Enum):
	ROCRATE = "ROCRATE"
	SOFTWARE = "SOFTWARE"
	DATASET = "DATASET"
	COMPUTATION = "COMPUTATION"
	ANNOTATION = "ANNOTATION"
	EXPERIMENT = "EXPERIMENT"
	PATIENT = "PATIENT"
	CREATIVE_WORK = "CREATIVE_WORK"
	SAMPLE = "SAMPLE"
	SCHEMA = "SCHEMA"
	BIO_CHEM_ENTITY = "BIO_CHEM_ENTITY"
	MEDICAL_CONDITION = "MEDICAL_CONDITION"
	PERSON = "PERSON"
	ORGANIZATION = "ORGANIZATION"
	DEFINED_TERM = "DEFINED_TERM"
	NONE = "NONE"


In [35]:
class IdentifiersSQL(Base):
	__tablename__ = 'identifier'
	__table_args__ = {"extend_existing": True}  
	guid: str = sa.Column('guid', sa.String, primary_key=True)
	name: str = sa.Column('name', sa.String)
	metadataType: Mapped[MetadataTypeEnumSQL] = mapped_column(sa.Enum(MetadataTypeEnumSQL))



/tmp/ipykernel_78113/4254617747.py:1: SAWarning: This declarative base already contains a class with the same class name and module name as __main__.IdentifiersSQL, and will be replaced in the string-lookup table.
  class IdentifiersSQL(Base):


In [5]:
# Other List Tables

class MetadataTypeSQL():
	pass

class D4DCorrectionsSQL():
	pass

class FunderSQL():
	pass

class CitationSQL():
	pass

class AssociatedPublicationsSQL():
	pass

class AdditionalPropertySQL():
	pass

class EthicalReviewSQL():
	pass

class IRBSQL():
	pass

class AboutSQL():
	pass

class RAIDataLimitationsSQL():
	pass

class RAIDataBiasesSQL():
	pass

class RAIDataUseCasesSQL():
	pass

class RAIDataReleaseMaintenancePlanSQL():
	pass

class RAIDataCollectionPlanSQL():
	pass

class RAIDataCollectionSQL():
	pass


class RAIDataCollectionTypeSQL():
	pass

class RAIDataCollectionMissingDataSQL():
	pass

class RAIDataCollectionRawDataSQL():
	pass

class RAIDataCollectionTimeframeSQL():
	pass

class RAIDataImputationProtocolSQL():
	pass

class RAIDataManipulationProtocolSQL():
	pass

class RAIDataPreprocessingProtocolSQL():
	pass

class RAIDataAnnotationProtocolSQL():
	pass

class RAIDataAnnotationPlatformSQL():
	pass

class RAIDataAnnotationAnalysisSQL():
	pass

class RAIPersonalSensitiveInformationSQL():
	pass

class RAIDataSocialImpactSQL():
	pass

class RAIAnnotationsPerItemSQL():
	pass

class RAIMachineAnnotationToolsSQL():
	pass

class CompletenessSQL():
	pass

class ProhibitedUsesSQL():
	pass

In [6]:
class KeywordSQL(Base):
	__tablename__ = 'keyword_table'
	__table_args__ = {"extend_existing": True}
	id: Mapped[int] = mapped_column(primary_key=True)
	guid: Mapped[str] = mapped_column(sa.ForeignKey("rocrate.guid")) 
	keywordValue: Mapped["ROCrateMetadataElemSQL"] = relationship()
	
class AuthorSQL(Base):
	__tablename__ = 'author'
	__table_args__ = {"extend_existing": True}  
	id: Mapped[int] = mapped_column(primary_key=True)
	guid: Mapped[str] = mapped_column(sa.ForeignKey("rocrate.guid")) 
	name: Mapped["ROCrateMetadataElemSQL"] = relationship()
	orcid: Mapped[Optional[str]] = mapped_column(default=None)

class MembershipSQL(Base):
	__tablename__ = 'membership'
	__table_args__ = {"extend_existing": True}  
	id: Mapped[int] = mapped_column(primary_key=True)
	parentGUID: Mapped[str] = mapped_column(sa.ForeignKey("rocrate.guid")) 
	parentType: Mapped[MetadataTypeEnumSQL] = mapped_column(sa.Enum(MetadataTypeEnumSQL))
	childGUID: Mapped["ROCrateMetadataElemSQL"] = relationship()
	childType: Mapped[MetadataTypeEnumSQL] = mapped_column(sa.Enum(MetadataTypeEnumSQL))


class ROCrateMetadataElemSQL(Base):
	__tablename__ = 'rocrate'
	__table_args__ = {"extend_existing": True}  
	id: Mapped[int] = mapped_column(primary_key=True)
	guid: str = sa.Column('guid', sa.String)
	name: str = sa.Column('name', sa.String)
	description: str = sa.Column('description', sa.String)
	keywords: Mapped[List["KeywordSQL"]] = relationship(back_populates="keywordValue")
	authors: Mapped[List["AuthorSQL"]] = relationship(back_populates="name")
	hasPart: Mapped[List["MembershipSQL"]] = relationship(back_populates="childGUID")

	# is part of add to membership table 
	#isPartOf: Mapped[Optional[List["MembershipSQL"]]] = relationship(back_populates="parentGUID")
	#datePublished: datetime
	#about: str
	#publisher: str

class EntitySQL():
	__table_args__ = {"extend_existing": True}  
	id: Mapped[int] = mapped_column(primary_key=True)
	guid: Mapped[str] = sa.Column('guid', sa.String)
	name: Mapped[str] = sa.Column('name', sa.String)
	contentURL: Mapped[Optional[str]] = sa.Column('contentURL', sa.String, default=None)


class DatasetSQL(EntitySQL, Base):
	__tablename__ = 'dataset'


class SoftwareSQL(EntitySQL, Base):
	__tablename__ = 'software'


class ComputationSQL(EntitySQL, Base):
	__tablename__ = 'computation'


In [ ]:
testGUID = "ark:59853/test"
testChildGUID= "ark:59853/test-member"

KeywordSQL(guid=testGUID, keywordValue='data')

# Create a test ROCrate with Keywords 
testElem = ROCrateMetadataElemSQL(
	guid= testGUID,
	name="Test ROCrate",
	description="A template ROCrate",
	keywords = [
		KeywordSQL(guid=testGUID, keywordValue='data'),
		KeywordSQL(guid=testGUID, keywordValue='test')
	],
	authors= [
		AuthorSQL(guid=testGUID, name="Max Levinson", orcid="https://orcid.org/0000-0003-0384-8499")
	],
	
	hasPart = [
		MembershipSQL(
			parentGUID=testGUID, 
			parentType=MetadataTypeEnumSQL.ROCRATE,
			childGUID=testChildGUID, 
			childType=MetadataTypeEnumSQL.DATASET
			)	
	]

)

## Converting `fairscape_models` into SQLAlchemy ORM Models

1. Load an Example ROCrate
2. Need to Create a Class to Contain all Records to create
 - Want to simply return the objects to create

In [10]:
dataversePath = pathlib.Path("/mnt/data/Dataverse")
releasePath = dataversePath / 'October2025'

# grab a single rocrate
cratePath = releasePath / 'cm4ai_ifimages_MDA-MB-468_paclitaxel_october_2025.zip'

In [ ]:

def findCrateRootMetadata(
	cratePath: pathlib.Path,
	)->bytes | None:
	""" Returns a 
	"""

	with zipfile.ZipFile(str(cratePath), 'r') as zip_ref:
		namelist = zip_ref.namelist()

		if 'ro-crate-metadata.json' in namelist:
			return zip_ref.read('ro-crate-metadata.json')

		#find root subfolder
		subfolders = [ elem for elem in namelist if elem.endswith("/") and elem.count("/") == 1]

		#	TODO exception for multiple root crates in 
		if len(subfolders) != 1:
			raise Exception

		crateZipPath = subfolders[0] + "ro-crate-metadata.json"

		# TODO exception
		if crateZipPath not in namelist:
			raise Exception

		# read the content	
		return zip_ref.read(crateZipPath)


def readCrate(
	cratePath: pathlib.Path,
	) -> ROCrateV1_2 | None:
	""" Given a path to a zipped crate read the rocrate json into an ROCrateV1_2"""
	rootCrateJSON = findCrateRootMetadata(cratePath)
	#rootCrateDict = json.loads(rootCrateJSON)

	try:
		return ROCrateV1_2.model_validate_json(rootCrateJSON)
	except Exception as e:
		return e.errors()

In [12]:
inputCrate = readCrate(cratePath)

In [14]:
inputCrate.metadataGraph[1]

ROCrateMetadataElem(guid='ark:59853/rocrate-paclitaxel-if-data-release', metadataType=['Dataset', 'https://w3id.org/EVI#ROCrate'], conformsTo=IdentifierValue(guid='https://w3id.org/fairscape/profile/0.1'), name='Paclitaxel IF Images', description='This data set displays the spatial localization of 464 proteins of interest in cells of the breast cancer cell line MDA-MB-468 treated with paclitaxel as imaged by immunofluorescence-based staining (ICC-IF) and confocal microscopy in the Lundberg Lab at Stanford University, as part of the Cell Maps for Artificial Intelligence (CM4AI; CM4AI.org) project. Nuclei were stained with DAPI (blue channel); endoplasmic reticulum with a calreticulin antibody (yellow channel); microtubules with tubulin antibody (red channel); and antibody against protein of interest (green channel). \n\nThis data is Copyright (c) 2025 The Board of Trustees of the Leland Stanford Junior University. It is licensed for reuse under Creative Commons Attribution ShareAlike No

In [15]:
len(inputCrate.metadataGraph)

22578

In [23]:
testCrateElem = inputCrate.metadataGraph[1]


In [ ]:
# use dumping ROCrateElem, use **kwargs to make ROCrateMetadataElem

In [49]:

def determineMetadataType(inputType: str | List[str])-> MetadataTypeEnumSQL:
	if isinstance(inputType, str):
		inputType = [inputType]

	# TODO rewrite
	if any(['ROCrate' in elem for elem in inputType]):
		return MetadataTypeEnumSQL.ROCRATE
	if any(['Dataset' in elem for elem in inputType]):
		return MetadataTypeEnumSQL.DATASET
	if any(['Software' in elem for elem in inputType]):
		return MetadataTypeEnumSQL.SOFTWARE
	if any(['Computation' in elem for elem in inputType]):
		return MetadataTypeEnumSQL.COMPUTATION
	if any(['Schema' in elem for elem in inputType]):
		return MetadataTypeEnumSQL.SCHEMA
	if any(['BioChemEntity' in elem for elem in inputType]):
		return MetadataTypeEnumSQL.BIO_CHEM_ENTITY

def convertROCrateMetadataToSQL(
	inputCrateElem: ROCrateMetadataElem
	)-> ROCrateMetadataElemSQL:
	""" Convert Basic ROCrateMetadata Elem to ROCrateMetadataElemSQL 
	TODO: Doesn't Handle hasPart relation, requires to look at the whole ROCrate
	"""

	authorOutput = []
	for auth in inputCrateElem.author:
		if isinstance(auth, str):
			authorOutput.append(
				AuthorSQL(
					guid=inputCrateElem.guid, 
					name=auth
				)	
			)
		if isinstance(auth, IdentifierValue):
			authorOutput.append(
				AuthorSQL(
					guid=inputCrateElem.guid, 
					name=auth.name, 
					orcid=auth.guid
				)	
			)

		return ROCrateMetadataElemSQL(
		guid= inputCrateElem.guid,
		name= inputCrateElem.name,
		description= inputCrateElem.description,
		keywords = [
			KeywordSQL(
				guid = inputCrateElem.guid, 
				keywordValue = keywordValue
				)
			for keywordValue in inputCrateElem.keywords
		],
		authors= 	authorOutput,	
		hasPart = []
	)


def getRootCrateMembership(inputCrate: ROCrateV1_2, rootCrateGUID: str) -> List[MembershipSQL]:
	""" Returns membership values for root ROCrateMetadataElemSQL.hasPart
	"""
	membershipRecords = []
	for metadataElem in inputCrate.metadataGraph:
		if isinstance(metadataElem, ROCrateMetadataFileElem):
			pass
		elif isinstance(metadataElem, ROCrateMetadataElem):
		# TODO: handle nested rocrates
			pass
		else:
			# create hasPart references
			# child type 
			membershipRecords.append(MembershipSQL(
				parentGUID= rootCrateGUID, 
				parentType=MetadataTypeEnumSQL.ROCRATE,
				childGUID=metadataElem.guid, 
				childType= determineMetadataType(metadataElem.metadataType)
				)	
			)
	return membershipRecords

In [50]:
convertedMetadataElem = convertROCrateMetadataToSQL(testCrateElem)
rootCrateGUID = convertedMetadataElem.guid
convertedMetadataElem.hasPart = getRootCrateMembership(inputCrate, rootCrateGUID)


In [51]:
convertedMetadataElem

In [30]:
testMetadataType = inputCrate.metadataGraph[10].metadataType

In [ ]:
crateElem.metadataType

['Dataset', 'https://w3id.org/EVI#ROCrate']

In [ ]:
# for every element in the metadata graph


22576

In [47]:
convertedMetadataElem.hasPart = membershipRecords

In [ ]:
# ROCrate Elem
crateElem.metadataType

['Dataset', 'https://w3id.org/EVI#ROCrate']

In [ ]:
# Software Elem


In [ ]:
# for every crate elem

In [ ]:
# fairscape_models -> SQLAlchemy
# have method for orm to_orm()

In [ ]:
# SQLAlchemy Models -> fairscape_models 

In [8]:
# create an engine
engine = sa.create_engine("sqlite:///test.db")

In [9]:
# create table 
Base.metadata.create_all(engine)

In [10]:
session = sa.orm.Session(engine)

In [11]:
session.add(testElem)

In [12]:
session.commit()

IntegrityError: (sqlite3.IntegrityError) UNIQUE constraint failed: rocrate.guid
[SQL: INSERT INTO rocrate (guid, name, description) VALUES (?, ?, ?)]
[parameters: ('ark:59853/test', 'Test ROCrate', 'A template ROCrate')]
(Background on this error at: https://sqlalche.me/e/20/gkpj)

In [13]:
query = sa.select(KeywordSQL).where(KeywordSQL.guid.in_([testGUID]))

In [ ]:
query

In [25]:
queryResults = session.execute(query)

In [28]:
queryResults.all()

[(<__main__.KeywordSQL object at 0x715126f06490>,),
 (<__main__.KeywordSQL object at 0x715126f06710>,)]